# MaaS Advanced Features

This notebook explores MaaS platform features beyond basic inference:

1. **Subscriptions & Rate Limits** — Configure token rate limit policies per group
2. **API Key Management** — Create, list, and revoke API keys
3. **Monitoring & Observability** — Query Prometheus metrics, check usage
4. **Access Control** — Test policy-based model access

**Prerequisites:**
- MaaS enabled with models and MCP servers registered (`2_enable_maas.ipynb` completed)
- Model serving and MCP tested (`3_test_model_serving.ipynb`, `4_test_mcp_servers.ipynb`)
- Cluster-admin or equivalent permissions for subscription/policy configuration

In [ ]:
import subprocess
import json
import time
import os

result = subprocess.run(
    ["kubectl", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()
MAAS_HOST = f"https://maas-api.apps.{CLUSTER_DOMAIN}"

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

print(f"✅ MaaS Gateway: {MAAS_HOST}")
print(f"   OCP User: {subprocess.run(['oc', 'whoami'], capture_output=True, text=True).stdout.strip()}")

---
## 1. Subscriptions & Token Rate Limits

MaaS uses **MaaSSubscription** CRDs to define per-group token rate limits.
A user must have both a matching **MaaSAuthPolicy** (access) and **MaaSSubscription** (quota) to use a model.

```
MaaSAuthPolicy  → Who can access which models
MaaSSubscription → How much quota they get (tokens per window)
```

### 1.1 View Existing Subscriptions and Policies

In [ ]:
%%bash
echo "=== MaaSModelRefs (registered models) ==="
kubectl get maasmodelref -A 2>/dev/null || echo "No MaaSModelRef resources found"

echo ""
echo "=== MaaSAuthPolicies (access control) ==="
kubectl get maasauthpolicy -n models-as-a-service 2>/dev/null || echo "No MaaSAuthPolicy resources found"

echo ""
echo "=== MaaSSubscriptions (rate limits) ==="
kubectl get maassubscription -n models-as-a-service 2>/dev/null || echo "No MaaSSubscription resources found"

echo ""
echo "=== Generated Kuadrant Policies ==="
echo "AuthPolicies:"
kubectl get authpolicy -A -l maas.opendatahub.io/model 2>/dev/null || echo "  None"
echo "TokenRateLimitPolicies:"
kubectl get tokenratelimitpolicy -A -l maas.opendatahub.io/model 2>/dev/null || echo "  None"

### 1.2 View Subscription Details

Inspect the rate limit configuration of existing subscriptions.

In [ ]:
%%bash
echo "=== MaaSSubscription Details ==="
for sub in $(kubectl get maassubscription -n models-as-a-service -o jsonpath='{.items[*].metadata.name}' 2>/dev/null); do
    echo ""
    echo "--- ${sub} ---"
    kubectl get maassubscription ${sub} -n models-as-a-service -o json 2>/dev/null | \
        python3 -c "
import sys, json
data = json.load(sys.stdin)
spec = data.get('spec', {})
print(f'  Priority: {spec.get(\"priority\", \"N/A\")}')
print(f'  Owner Groups: {[g[\"name\"] for g in spec.get(\"owner\", {}).get(\"groups\", [])]}')
for ref in spec.get('modelRefs', []):
    limits = ref.get('tokenRateLimits', [])
    for lim in limits:
        print(f'  Model: {ref[\"namespace\"]}/{ref[\"name\"]}  →  {lim[\"limit\"]} tokens / {lim[\"window\"]}')
"
done

### 1.3 Create a Test Subscription with Low Rate Limit

Create a subscription with a deliberately low token limit to observe rate limiting behavior.

**⚠️ Adjust `MODEL_NAME` and `MODEL_NS` for your environment.**

In [ ]:
%%bash
# Adjust these for your environment
MODEL_NS=${MODEL_NS:-llm}
MODEL_REF=$(kubectl get maasmodelref -n ${MODEL_NS} -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$MODEL_REF" ]; then
    echo "⚠️  No MaaSModelRef found in namespace '${MODEL_NS}'. Adjust MODEL_NS."
    exit 0
fi

echo "Using model ref: ${MODEL_NS}/${MODEL_REF}"
echo ""

# Create a group for testing
oc adm groups new lab-test-users 2>/dev/null || true
oc adm groups add-users lab-test-users $(oc whoami) 2>/dev/null || true
echo "✅ Added $(oc whoami) to lab-test-users group"

# Create access policy
kubectl apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSAuthPolicy
metadata:
  name: lab-test-access
  namespace: models-as-a-service
spec:
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
  subjects:
    groups:
      - name: lab-test-users
    users: []
EOF

echo ""

# Create subscription with low limit (50 tokens per minute)
kubectl apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: lab-test-subscription
  namespace: models-as-a-service
spec:
  owner:
    groups:
      - name: lab-test-users
    users: []
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
      tokenRateLimits:
        - limit: 50
          window: 1m
  priority: 5
EOF

echo ""
echo "✅ Created lab-test-subscription: 50 tokens/minute"
echo "   Wait ~30s for policies to reconcile..."

### 1.4 Test Rate Limiting with Low Limit

Send requests until the 50 token/minute limit is hit. Expect 429 (Too Many Requests).

In [ ]:
%%bash
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.apps.${CLUSTER_DOMAIN}"

# Create an API key bound to our test subscription
API_KEY_RESP=$(curl -sSk \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" \
  -X POST \
  -d '{"name": "rate-limit-test", "expiresIn": "1h", "subscription": "lab-test-subscription"}' \
  "${HOST}/maas-api/v1/api-keys")

TEST_KEY=$(echo $API_KEY_RESP | python3 -c "import sys,json; print(json.load(sys.stdin).get('key',''))")

if [ -z "$TEST_KEY" ]; then
    echo "⚠️  Could not create API key. Response:"
    echo $API_KEY_RESP | python3 -m json.tool
    exit 0
fi

echo "API Key: ${TEST_KEY:0:20}..."
echo ""

# Get model URL
MODELS_JSON=$(curl -sSk "${HOST}/maas-api/v1/models" \
  -H "Authorization: Bearer ${TEST_KEY}" \
  -H "Content-Type: application/json")
MODEL_NAME=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['id'] if d.get('data') else '')")
MODEL_URL=$(echo $MODELS_JSON | python3 -c "import sys,json; d=json.load(sys.stdin); print(d['data'][0]['url'] if d.get('data') else '')")

echo "Testing rate limit (50 tokens/min) on ${MODEL_NAME}..."
echo ""

for i in $(seq 1 8); do
  HTTP_CODE=$(curl -sSk -o /dev/null -w "%{http_code}" \
    -H "Authorization: Bearer ${TEST_KEY}" \
    -H "Content-Type: application/json" \
    -d "{\"model\": \"${MODEL_NAME}\", \"messages\": [{\"role\": \"user\", \"content\": \"Write a haiku about coding.\"}], \"max_tokens\": 30}" \
    "${MODEL_URL}/v1/chat/completions")
  
  if [ "$HTTP_CODE" = "429" ]; then
    echo "  Request $i: HTTP $HTTP_CODE ← Rate limit hit! ✅"
  else
    echo "  Request $i: HTTP $HTTP_CODE"
  fi
done

### 1.5 Increase Rate Limit

Update the subscription to allow more tokens and re-test.

In [ ]:
%%bash
MODEL_NS=${MODEL_NS:-llm}
MODEL_REF=$(kubectl get maasmodelref -n ${MODEL_NS} -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$MODEL_REF" ]; then
    echo "⚠️  No MaaSModelRef found. Skip."
    exit 0
fi

kubectl apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: lab-test-subscription
  namespace: models-as-a-service
spec:
  owner:
    groups:
      - name: lab-test-users
    users: []
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
      tokenRateLimits:
        - limit: 10000
          window: 1m
  priority: 5
EOF

echo ""
echo "✅ Updated lab-test-subscription: 50 → 10,000 tokens/minute"
echo ""

# Wait for policy to reconcile
echo "Waiting for TokenRateLimitPolicy to update..."
sleep 10

kubectl get tokenratelimitpolicy -A -l maas.opendatahub.io/model=${MODEL_REF} 2>/dev/null
echo ""
echo "Now re-run the rate limit test — requests should succeed with the higher limit."

---
## 2. API Key Management

MaaS provides full lifecycle management for API keys: create, list, inspect, and revoke.

### 2.1 Create API Keys with Different Expirations

In [ ]:
import urllib.request
import ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

keys_created = []

key_configs = [
    {"name": "short-lived", "description": "Expires in 1 hour", "expiresIn": "1h"},
    {"name": "standard", "description": "Expires in 30 days", "expiresIn": "30d"},
    {"name": "ephemeral-demo", "ephemeral": True, "expiresIn": "30m"},
]

print("Creating API keys with different expirations:")
print("=" * 70)

for config in key_configs:
    data = json.dumps(config).encode()
    req = urllib.request.Request(
        f"{MAAS_HOST}/maas-api/v1/api-keys",
        data=data,
        headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req, context=ctx) as resp:
            result = json.loads(resp.read())
            key_id = result.get("id", "")
            key = result.get("key", "")
            expires = result.get("expiresAt", "")
            sub = result.get("subscription", "")
            keys_created.append({"id": key_id, "key": key, "name": config.get("name", "ephemeral")})
            print(f"  {config.get('name', 'ephemeral'):<20} key={key[:20]}...  expires={expires}  sub={sub}")
    except Exception as e:
        print(f"  {config.get('name', 'ephemeral'):<20} ❌ {str(e)[:60]}")

print(f"\n✅ Created {len(keys_created)} API keys")

### 2.2 List Active API Keys

In [ ]:
search_data = json.dumps({"status": "active", "limit": 20, "includeEphemeral": True}).encode()
req = urllib.request.Request(
    f"{MAAS_HOST}/maas-api/v1/api-keys/search",
    data=search_data,
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        keys_list = json.loads(resp.read())

    print("Active API Keys")
    print("=" * 80)
    print(f"{'Name':<20} {'ID':<38} {'Subscription':<25} {'Expires'}")
    print("-" * 80)

    for key in keys_list if isinstance(keys_list, list) else keys_list.get("items", keys_list.get("data", [])):
        name = key.get("name", "(ephemeral)")
        print(f"{name:<20} {key.get('id', ''):<38} {key.get('subscription', ''):<25} {key.get('expiresAt', '')}")
except Exception as e:
    print(f"❌ Error listing keys: {e}")

### 2.3 Revoke an API Key

In [ ]:
if keys_created:
    key_to_revoke = keys_created[0]  # Revoke the short-lived key
    key_id = key_to_revoke["id"]
    key_name = key_to_revoke["name"]

    print(f"Revoking key '{key_name}' (ID: {key_id})...")

    req = urllib.request.Request(
        f"{MAAS_HOST}/maas-api/v1/api-keys/{key_id}",
        headers={"Authorization": f"Bearer {OC_TOKEN}"},
        method="DELETE"
    )
    try:
        with urllib.request.urlopen(req, context=ctx) as resp:
            print(f"✅ Key '{key_name}' revoked successfully")
    except Exception as e:
        print(f"❌ Error: {e}")

    # Verify it's revoked
    print(f"\nTesting revoked key...")
    revoked_key = key_to_revoke["key"]
    req = urllib.request.Request(
        f"{MAAS_HOST}/maas-api/v1/models",
        headers={"Authorization": f"Bearer {revoked_key}", "Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req, context=ctx) as resp:
            print(f"⚠️  Key still works (HTTP {resp.status}) — may be cached briefly")
    except urllib.error.HTTPError as e:
        print(f"✅ Revoked key rejected (HTTP {e.code})")
else:
    print("No keys to revoke — run the creation cell first.")

---
## 3. Monitoring & Observability

MaaS exposes metrics through Prometheus. Key metrics include:

| Metric | Source | What It Measures |
|--------|--------|------------------|
| `authorized_calls` | Limitador | Successful requests (within rate limit) |
| `authorized_hits` | Limitador | Token usage (authorized tokens consumed) |
| `limited_calls` | Limitador | Rejected requests (rate limit exceeded) |
| `auth_server_evaluations_total` | Authorino | Auth evaluation count |
| `auth_server_response_status` | Authorino | Auth success/deny breakdown |

### 3.1 Check Observability Prerequisites

In [ ]:
%%bash
echo "=== User Workload Monitoring ==="
UWM_PODS=$(kubectl get pods -n openshift-user-workload-monitoring --no-headers 2>/dev/null | wc -l)
if [ "$UWM_PODS" -gt 0 ]; then
    echo "✅ User Workload Monitoring active ($UWM_PODS pods)"
    kubectl get pods -n openshift-user-workload-monitoring --no-headers 2>/dev/null
else
    echo "⚠️  User Workload Monitoring not active"
    echo "   Enable it with:"
    echo '   kubectl apply -f - <<EOF'
    echo '   apiVersion: v1'
    echo '   kind: ConfigMap'
    echo '   metadata:'
    echo '     name: cluster-monitoring-config'
    echo '     namespace: openshift-monitoring'
    echo '   data:'
    echo '     config.yaml: |'
    echo '       enableUserWorkload: true'
    echo '   EOF'
fi

echo ""
echo "=== Kuadrant Observability ==="
PODMON=$(kubectl get podmonitor -n kuadrant-system --no-headers 2>/dev/null | wc -l)
if [ "$PODMON" -gt 0 ]; then
    echo "✅ Kuadrant PodMonitor found"
    kubectl get podmonitor -n kuadrant-system --no-headers 2>/dev/null
else
    echo "⚠️  Kuadrant PodMonitor not found"
    echo "   Enable with: kubectl patch kuadrant kuadrant -n kuadrant-system --type=merge -p '{\"spec\":{\"observability\":{\"enable\":true}}}'"
fi

echo ""
echo "=== Limitador Pods ==="
kubectl get pods -n kuadrant-system -l app=limitador --no-headers 2>/dev/null || echo "  No Limitador pods found"

### 3.2 Query Prometheus Metrics

Query the OpenShift Thanos endpoint for MaaS usage metrics.

In [ ]:
import urllib.parse

THANOS_HOST = f"https://thanos-querier-openshift-monitoring.{CLUSTER_DOMAIN}"

queries = {
    "Authorized Calls (total)": "sum(authorized_calls)",
    "Authorized Tokens (total)": "sum(authorized_hits)",
    "Rate-Limited Calls (total)": "sum(limited_calls)",
    "Limitador Up": "limitador_up",
}

print("MaaS Prometheus Metrics")
print("=" * 60)

for label, query in queries.items():
    try:
        url = f"{THANOS_HOST}/api/v1/query?query={urllib.parse.quote(query)}"
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {OC_TOKEN}"})
        with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
            data = json.loads(resp.read())
            results = data.get("data", {}).get("result", [])
            if results:
                value = results[0].get("value", ["", "N/A"])[1]
                print(f"  {label:<35} {value}")
            else:
                print(f"  {label:<35} (no data)")
    except Exception as e:
        print(f"  {label:<35} ❌ {str(e)[:40]}")

### 3.3 Rate Limit Metrics Over Time

Query the rate of authorized vs limited calls over the last 5 minutes.

In [ ]:
import urllib.parse

range_queries = {
    "Authorized calls/min (5m avg)": "rate(authorized_calls[5m]) * 60",
    "Limited calls/min (5m avg)": "rate(limited_calls[5m]) * 60",
    "Token consumption/min (5m avg)": "rate(authorized_hits[5m]) * 60",
}

print("Rate Metrics (5-minute window)")
print("=" * 60)

for label, query in range_queries.items():
    try:
        url = f"{THANOS_HOST}/api/v1/query?query={urllib.parse.quote(query)}"
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {OC_TOKEN}"})
        with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
            data = json.loads(resp.read())
            results = data.get("data", {}).get("result", [])
            if results:
                value = float(results[0].get("value", ["", "0"])[1])
                print(f"  {label:<40} {value:.2f}")
            else:
                print(f"  {label:<40} (no data)")
    except Exception as e:
        print(f"  {label:<40} ❌ {str(e)[:40]}")

### 3.4 Component Health Check

In [ ]:
%%bash
echo "MaaS Component Health"
echo "=" * 60

echo ""
echo "=== MaaS Gateway ==="
kubectl get gateway -n openshift-ingress maas-default-gateway -o jsonpath='{.status.conditions[*].type}={.status.conditions[*].status}' 2>/dev/null
echo ""

echo ""
echo "=== MaaS API Pods ==="
APP_NS=$(kubectl get ns redhat-ods-applications --no-headers 2>/dev/null && echo redhat-ods-applications || echo opendatahub)
kubectl get pods -n ${APP_NS} -l app.kubernetes.io/name=maas-api --no-headers 2>/dev/null || echo "  Not found"

echo ""
echo "=== Kuadrant ==="
kubectl get pods -n kuadrant-system --no-headers 2>/dev/null | head -5

echo ""
echo "=== Authorino ==="
kubectl get pods -n kuadrant-system -l app=authorino --no-headers 2>/dev/null || echo "  Not found"

echo ""
echo "=== Limitador ==="
kubectl get pods -n kuadrant-system -l app=limitador --no-headers 2>/dev/null || echo "  Not found"

echo ""
echo "=== Tenant CR ==="
kubectl get tenant -n models-as-a-service --no-headers 2>/dev/null || echo "  Not found"

---
## 4. Cleanup Test Resources

Remove the test subscription, policy, and group created in this notebook.

In [ ]:
%%bash
echo "Cleaning up lab test resources..."

kubectl delete maassubscription lab-test-subscription -n models-as-a-service 2>/dev/null && echo "✅ Deleted lab-test-subscription" || echo "  (not found)"
kubectl delete maasauthpolicy lab-test-access -n models-as-a-service 2>/dev/null && echo "✅ Deleted lab-test-access" || echo "  (not found)"
oc adm groups remove-users lab-test-users $(oc whoami) 2>/dev/null && echo "✅ Removed user from lab-test-users" || echo "  (not found)"

# Revoke remaining test keys
echo ""
echo "Bulk-revoking test API keys..."
CLUSTER_DOMAIN=$(kubectl get ingresses.config.openshift.io cluster -o jsonpath='{.spec.domain}')
HOST="https://maas-api.apps.${CLUSTER_DOMAIN}"
curl -sSk -X POST "${HOST}/maas-api/v1/api-keys/bulk-revoke" \
  -H "Authorization: Bearer $(oc whoami -t)" \
  -H "Content-Type: application/json" 2>/dev/null | python3 -m json.tool || echo "  (no keys to revoke)"

echo ""
echo "✅ Cleanup complete"

---
## Summary

| Feature | What We Tested |
|---------|----------------|
| **Subscriptions** | Created MaaSSubscription with token rate limits, tested enforcement |
| **Rate Limit Tuning** | Changed limit from 50 → 10,000 tokens/min, verified update |
| **API Key Lifecycle** | Created keys (short-lived, standard, ephemeral), listed, revoked |
| **Monitoring** | Queried Prometheus for authorized/limited calls and token usage |
| **Health Check** | Verified Gateway, Authorino, Limitador, MaaS API pod status |

### Key MaaS CRDs

| CRD | Namespace | Purpose |
|-----|-----------|--------|
| `MaaSModelRef` | Model namespace (e.g., `llm`) | Registers a model for MaaS |
| `MaaSAuthPolicy` | `models-as-a-service` | Grants group/user access to models |
| `MaaSSubscription` | `models-as-a-service` | Defines token rate limits per group |

### Reference

- [Quota & Access Configuration](https://opendatahub-io.github.io/models-as-a-service/latest/configuration-and-management/quota-and-access-configuration/)
- [API Key Management](https://opendatahub-io.github.io/models-as-a-service/latest/user-guide/api-key-management/)
- [Observability Dashboard](https://opendatahub-io.github.io/models-as-a-service/latest/advanced-administration/observability/)